# Propositional Logic — Practice (with solutions)

**Topic:** Session 9 — Knowledge Representation using Logics (Module M4, AIMLCZG557)  
**Companion notes:** `../../materials/session-9-knowledge-representation-using-logics.md`

This notebook is a set of **exercises with worked solutions** so you can practise.
For each exercise:

1. Read the problem.
2. **Try it yourself first** in the *Your attempt* cell (paper works too for the by-hand ones).
3. Then run / expand the **Solution** cell and compare.

Everything runs on a plain Anaconda install (only `itertools` and `pandas`, both standard).

---

In [ ]:
# Run me first — shared helpers used across the notebook
import itertools
import pandas as pd

pd.set_option('display.max_rows', 50)
print('Ready. pandas', pd.__version__)

## Exercise 1 — Is it a proposition?

A **proposition** is a *declarative* sentence that is either **True** or **False**, but not both.
For each statement below, decide whether it is a proposition, and if so give its truth value.

| # | Statement |
|---|-----------|
| a | The sun rises in the East. |
| b | 1 + 1 = 2 |
| c | 'b' is a vowel. |
| d | Will you go to the office today? |
| e | x − 3 = 6 |
| f | Please close the door. |

### Your attempt
*(write a / b / c ... = proposition? truth value)*



### Solution

| # | Proposition? | Truth value | Why |
|---|-------------|-------------|-----|
| a | ✅ Yes | **True** | Declarative, definitely true. |
| b | ✅ Yes | **True** | Declarative, arithmetic fact. |
| c | ✅ Yes | **False** | Declarative — 'b' is a consonant. |
| d | ❌ No | — | It's a *question*, not declarative. |
| e | ❌ No | — | Truth depends on `x`; not fixed True/False. |
| f | ❌ No | — | It's a *command* (imperative), not declarative. |

**Takeaway:** questions, commands, and open statements with free variables are **not** propositions.

## Exercise 2 — Build a truth table

Build the full truth table for the compound sentence

$$ (P \Rightarrow Q) \;\wedge\; (\neg P \vee Q) $$

Then answer: is this sentence a **tautology** (true in every model), a **contradiction** (false in every model), or **contingent**?

In [ ]:
# Your attempt: fill in the body of `evaluate` and build the table.
# implies(p, q) is False only when p is True and q is False.

def implies(p, q):
    return (not p) or q   # TODO if you want to try: derive this yourself

# TODO: compute the sentence for a given P, Q
def evaluate(P, Q):
    return None

# rows = [(P, Q, evaluate(P,Q)) for P in (True,False) for Q in (True,False)]
# print(rows)

In [ ]:
# Solution
def implies(p, q):
    return (not p) or q

def evaluate(P, Q):
    return implies(P, Q) and ((not P) or Q)

rows = []
for P in (True, False):
    for Q in (True, False):
        rows.append({
            'P': P, 'Q': Q,
            'P⇒Q': implies(P, Q),
            '¬P∨Q': (not P) or Q,
            'sentence': evaluate(P, Q),
        })
df = pd.DataFrame(rows)
display(df)

col = df['sentence']
if col.all():
    print('=> TAUTOLOGY (true in every model)')
elif not col.any():
    print('=> CONTRADICTION (false in every model)')
else:
    print('=> CONTINGENT (sometimes true, sometimes false)')

**Answer:** it is a **tautology**. In fact `P ⇒ Q` and `¬P ∨ Q` are *logically equivalent*, so their conjunction equals `P ⇒ Q`, which... is not itself a tautology — but the equivalence means the two columns are always equal, and the sentence equals `P ⇒ Q`. Run the cell: the `sentence` column matches `P⇒Q` exactly, confirming `P ⇒ Q ≡ ¬P ∨ Q` (a key rewrite used in CNF conversion).

> If you expected "tautology" and got a contingent result, that's the point of the exercise: the sentence is **not** always true — but it *reveals the equivalence* `P ⇒ Q ≡ ¬P ∨ Q`.

## Exercise 3 — Entailment by model checking (TT-Entails)

Recall the Wumpus knowledge base from the notes:

```
R1 : ¬P1,1
R2 : B1,1 ⟺ (P1,2 ∨ P2,1)
R3 : B2,1 ⟺ (P1,1 ∨ P2,2 ∨ P3,1)
R4 : ¬B1,1
R5 :  B2,1
```

**Task:** implement `tt_entails(kb, query)` by *model checking* — enumerate every assignment of the
symbols, keep the models where **all** KB rules hold, and check whether the query is true in **all** of them.

Use it to verify: **is `¬P1,2` entailed?** (The notes say yes.) Also test `¬P2,2` (the notes say we *cannot* infer it).

In [ ]:
# Your attempt.
# Represent each rule as a function taking a dict `m` of {symbol: bool}.
# Example: R1 = lambda m: not m['P11']

symbols = ['P11','P12','P21','P22','P31','B11','B21']

def iff(a, b):  return a == b
def imp(a, b):  return (not a) or b

KB = [
    # TODO: translate R1..R5 into lambdas over m
]

def tt_entails(kb, query):
    # TODO: enumerate models, return True iff query holds in every model where all kb rules hold
    return None

In [ ]:
# Solution
symbols = ['P11','P12','P21','P22','P31','B11','B21']

def iff(a, b):  return a == b
def imp(a, b):  return (not a) or b

KB = [
    lambda m: not m['P11'],                                   # R1: ¬P1,1
    lambda m: iff(m['B11'], m['P12'] or m['P21']),            # R2
    lambda m: iff(m['B21'], m['P11'] or m['P22'] or m['P31']),# R3
    lambda m: not m['B11'],                                   # R4: ¬B1,1
    lambda m: m['B21'],                                       # R5:  B2,1
]

def models(symbols):
    for combo in itertools.product([False, True], repeat=len(symbols)):
        yield dict(zip(symbols, combo))

def tt_entails(kb, query):
    kb_models = [m for m in models(symbols) if all(rule(m) for rule in kb)]
    if not kb_models:
        return None, []          # KB unsatisfiable
    entailed = all(query(m) for m in kb_models)
    return entailed, kb_models

ans1, kb_models = tt_entails(KB, lambda m: not m['P12'])   # query ¬P1,2
ans2, _         = tt_entails(KB, lambda m: not m['P22'])   # query ¬P2,2

print('Number of models where KB is true:', len(kb_models))
print('KB entails  ¬P1,2 ?', ans1)
print('KB entails  ¬P2,2 ?', ans2)
print()
print('The models where the KB holds:')
display(pd.DataFrame(kb_models)[symbols])

**Expected result:** the KB is satisfied by exactly **3** models. `¬P1,2` is `True` in all three → **entailed** (no pit in `[1,2]`). `¬P2,2` is `True` in some and `False` in others → **not entailed** (we can't decide `[2,2]`).

Complexity note: with `n` symbols this checks `2ⁿ` models — `O(2ⁿ)` time, `O(n)` space. That exponential cost motivates resolution & DPLL.

## Exercise 4 — Convert to CNF (by hand)

Convert `B1,1 ⟺ (P1,2 ∨ P2,1)` into **Conjunctive Normal Form** (a conjunction of clauses).
Show every step and name the rule you used.

*Rules you'll need:* biconditional elimination, implication elimination (`a⇒b ≡ ¬a∨b`), De Morgan, distributivity of `∨` over `∧`.

### Your attempt

1. 
2. 
3. 
4. 

### Solution

| Step | Result | Rule |
|------|--------|------|
| 1 | `(B1,1 ⇒ (P1,2 ∨ P2,1)) ∧ ((P1,2 ∨ P2,1) ⇒ B1,1)` | biconditional elimination |
| 2 | `(¬B1,1 ∨ P1,2 ∨ P2,1) ∧ (¬(P1,2 ∨ P2,1) ∨ B1,1)` | implication elimination |
| 3 | `(¬B1,1 ∨ P1,2 ∨ P2,1) ∧ ((¬P1,2 ∧ ¬P2,1) ∨ B1,1)` | De Morgan on `¬(P1,2 ∨ P2,1)` |
| 4 | `(¬B1,1 ∨ P1,2 ∨ P2,1) ∧ (¬P1,2 ∨ B1,1) ∧ (¬P2,1 ∨ B1,1)` | distribute `∨` over `∧` |

**Final CNF — three clauses:**

```
¬B1,1 ∨ P1,2 ∨ P2,1
¬P1,2 ∨ B1,1
¬P2,1 ∨ B1,1
```

These are exactly `R6`, `R7`, `R8` from the notes.

## Exercise 5 — Prove `¬P1,2` by resolution refutation

Resolution proves a query by **contradiction**: add the *negation* of the query to the clause set and
resolve until you derive the **empty clause `{}`** (= FALSE).

**Resolution rule:** clauses `(A ∨ ℓ)` and `(B ∨ ¬ℓ)` resolve to `(A ∨ B)`.

**Task:** implement a tiny resolution engine. Represent a clause as a Python `frozenset` of literals,
where a literal is `'P12'` (positive) or `'~P12'` (negative). Then feed it the Wumpus CNF clauses plus the
negated query `P12`, and check that it derives the empty clause.

In [ ]:
# Your attempt
def negate(lit):
    # TODO: 'P12' -> '~P12', '~P12' -> 'P12'
    ...

def resolve(ci, cj):
    # TODO: return the set of resolvents of two clauses (each a frozenset of literals)
    ...

def pl_resolution(clauses):
    # TODO: return True if the clause set is UNSAT (empty clause derivable)
    ...

In [ ]:
# Solution
def negate(lit):
    return lit[1:] if lit.startswith('~') else '~' + lit

def resolve(ci, cj):
    resolvents = set()
    for lit in ci:
        if negate(lit) in cj:
            new = (ci - {lit}) | (cj - {negate(lit)})
            # skip tautologies (clause containing x and ~x)
            if not any(negate(x) in new for x in new):
                resolvents.add(frozenset(new))
    return resolvents

def pl_resolution(clauses):
    clauses = set(clauses)
    while True:
        new = set()
        pairs = itertools.combinations(clauses, 2)
        for ci, cj in pairs:
            for r in resolve(ci, cj):
                if len(r) == 0:
                    return True          # empty clause => contradiction => UNSAT
                new.add(r)
        if new.issubset(clauses):
            return False                 # no new info => SAT (no contradiction)
        clauses |= new

# Wumpus KB in CNF (R1, R4, R5, R6..R12) as clauses:
kb_clauses = [
    frozenset({'~P11'}),                         # R1
    frozenset({'~B11'}),                         # R4
    frozenset({'B21'}),                          # R5
    frozenset({'~B11','P12','P21'}),             # R6
    frozenset({'~P12','B11'}),                   # R7
    frozenset({'~P21','B11'}),                   # R8
    frozenset({'~B21','P11','P22','P31'}),       # R9
    frozenset({'~P11','B21'}),                   # R10
    frozenset({'~P22','B21'}),                   # R11
    frozenset({'~P31','B21'}),                   # R12
]

# To prove ¬P1,2, add the NEGATED query (P1,2) and look for a contradiction:
proof_set = kb_clauses + [frozenset({'P12'})]
print('KB + assume(P1,2) is UNSAT  ?', pl_resolution(proof_set))
print('  -> contradiction found => ¬P1,2 is ENTAILED\n')

# Sanity check: KB alone should be satisfiable (no contradiction).
print('KB alone is UNSAT           ?', pl_resolution(kb_clauses), '(expected False)')

**Expected:** `KB + assume(P1,2)` is **UNSAT (True)** → the assumption `P1,2` is impossible → `¬P1,2` is entailed. The KB on its own is satisfiable (`False`), as it should be.

The shortest hand proof: `R4: ¬B1,1` resolves with `R7: ¬P1,2 ∨ B1,1` to give `¬P1,2`; that resolves with the assumed `P1,2` to give the empty clause `{}`.

## Exercise 6 — Apply the inference rules (name each step)

Given the KB below, prove the goal **`R` (I need to rest)** using named inference rules only
(Modus Ponens, And-Elimination, etc.). No truth tables.

```
1. Tired ∧ Sleepy            (given)
2. Tired ⟹ R                 (given)
```

Goal: `R`

### Your attempt

3. 
4. 

### Solution

| Step | Sentence | Rule |
|------|----------|------|
| 1 | `Tired ∧ Sleepy` | given |
| 2 | `Tired ⟹ R` | given |
| 3 | `Tired` | **And-Elimination** on 1 |
| 4 | `R` | **Modus Ponens** on 2 and 3 ∎ |

## Exercise 7 — Why propositional logic isn't enough (predicate logic)

Propositional logic has **limited expressive power**. Consider the English sentences:

- (a) *Every student who studies passes.*
- (b) *Some student is brilliant.*
- (c) *Pits cause a breeze in every adjacent square.*

**Task:** explain why plain propositional logic struggles with these, and write each in **predicate (first-order) logic** using quantifiers `∀` (for all) and `∃` (there exists).

### Your attempt

a. 
b. 
c. 

### Solution

**Why propositional logic struggles:** it has no notion of *objects*, *properties*, or *quantifiers*. To say "every student passes" you would need a separate proposition for each individual student, and to say "pits cause breezes in adjacent squares" you'd need one sentence per square (exactly the complaint in the notes). Predicate logic adds objects, predicates, and quantifiers so a single sentence covers all cases.

**Predicate-logic forms** (one reasonable rendering):

- (a) `∀x. (Student(x) ∧ Studies(x)) ⟹ Passes(x)`
- (b) `∃x. (Student(x) ∧ Brilliant(x))`
- (c) `∀x ∀y. (Pit(x) ∧ Adjacent(x, y)) ⟹ Breeze(y)`

That last one is a *single* sentence replacing the 16 per-square propositions — the whole reason we move on to predicate logic and inference by **forward/backward chaining**.

---
## Quick self-check checklist

- [ ] I can tell a proposition from a non-proposition.
- [ ] I can build a truth table and classify a sentence (tautology / contradiction / contingent).
- [ ] I can decide entailment by model checking and explain its `O(2ⁿ)` cost.
- [ ] I can convert a biconditional to CNF step by step.
- [ ] I can run a resolution refutation to the empty clause.
- [ ] I can name the inference rule at each step of a proof.
- [ ] I can say why we need predicate logic and write `∀` / `∃` sentences.

When all boxes are ticked, you've covered Session 9.